Step 1: Enriching the count json for easy lookup. Why? because I just want to look for the logs of those test files which were detected and has real assert calls. 

In [ ]:
"""
Java Assert Analyzer v3 + Batch Runner — single file
Handles .txt files containing Java code with optional markdown code fences.
Includes per-method assertion tracking, method line ranges.
Handles: JUnit 4, JUnit 5, TestNG, AssertJ, Hamcrest

Install:
    pip install tree-sitter tree-sitter-java
"""

import os
import re
import csv
import sys
import json
import inspect as _inspect
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

try:
    import tree_sitter_java as tsjava
    from tree_sitter import Language, Parser
except ImportError:
    sys.exit(
        "Missing deps. Run:\n"
        "  pip install tree-sitter tree-sitter-java"
    )

# ── Handle both old and new tree-sitter APIs ──────────────────────────────────
_lang_params = list(_inspect.signature(Language.__init__).parameters.keys())
_OLD_API = "name" in _lang_params

if _OLD_API:
    JAVA_LANG = Language(tsjava.language(), "java")
    PARSER    = Parser()
    PARSER.set_language(JAVA_LANG)
else:
    JAVA_LANG = Language(tsjava.language())
    PARSER    = Parser(JAVA_LANG)


# =============================================================================
# CONFIGURATION
# =============================================================================

DATASETS = {
    "compiled_pre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre",
    "executed_pre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/execution_pre",
    "detected_bre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/detected_bre",
}

OUTPUT_DIR = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o"

FILE_PATTERNS = ["*_prompt.txt", "*.java"]

RESULTS_CACHE = f"{OUTPUT_DIR}/all_results_cache.json"


# =============================================================================
# FRAMEWORK DEFINITIONS
# =============================================================================

FRAMEWORKS = {
    "junit4": {
        "imports": ["org.junit.Assert", "org.junit.Assert.*"],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertArrayEquals", "assertThat", "fail",
        },
        "qualified_class": "Assert",
    },
    "junit5": {
        "imports": [
            "org.junit.jupiter.api.Assertions",
            "org.junit.jupiter.api.Assertions.*",
        ],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertArrayEquals", "assertThrows", "assertDoesNotThrow",
            "assertTimeout", "assertTimeoutPreemptively",
            "assertIterableEquals", "assertLinesMatch", "assertAll", "fail",
        },
        "qualified_class": "Assertions",
    },
    "testng": {
        "imports": [
            "org.testng.Assert",
            "org.testng.Assert.*",
            "org.testng.AssertJUnit",
            "org.testng.AssertJUnit.*",
            "org.testng.asserts.SoftAssert",
        ],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertEqualsNoOrder", "assertThrows", "expectThrows", "fail",
        },
        "qualified_class": "Assert",
        "soft_classes": {"SoftAssert"},
    },
    "assertj": {
        "imports": [
            "org.assertj.core.api.Assertions",
            "org.assertj.core.api.Assertions.*",
            "org.assertj.core.api.SoftAssertions",
            "org.assertj.core.api.BDDAssertions",
            "org.assertj.core.api.BDDAssertions.*",
        ],
        "static_methods": {
            "assertThat", "assertThatThrownBy", "assertThatCode",
            "assertThatExceptionOfType", "assertThatNoException",
            "assertThatObject", "assertThatList",
            "catchThrowable", "catchThrowableOfType", "fail",
            "then", "thenThrownBy",
        },
        "qualified_class": "Assertions",
        "fluent": True,
        "soft_classes": {"SoftAssertions", "BDDSoftAssertions", "JUnitSoftAssertions"},
    },
    "hamcrest": {
        "imports": ["org.hamcrest.MatcherAssert", "org.hamcrest.MatcherAssert.*"],
        "static_methods": {"assertThat"},
        "qualified_class": "MatcherAssert",
    },
}

_METHOD_TO_FRAMEWORKS: dict[str, list[str]] = defaultdict(list)
for _fw, _cfg in FRAMEWORKS.items():
    for _m in _cfg.get("static_methods", set()):
        _METHOD_TO_FRAMEWORKS[_m].append(_fw)

ALL_ASSERT_METHODS: set[str] = set(_METHOD_TO_FRAMEWORKS.keys())
ALL_QUALIFIED_CLASSES: set[str] = {
    cfg["qualified_class"] for cfg in FRAMEWORKS.values() if "qualified_class" in cfg
}
ALL_SOFT_CLASSES: set[str] = {
    c for cfg in FRAMEWORKS.values() for c in cfg.get("soft_classes", set())
}
SOFT_CLASS_TO_FRAMEWORK: dict[str, str] = {
    c: fw
    for fw, cfg in FRAMEWORKS.items()
    for c in cfg.get("soft_classes", set())
}


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class AssertCall:
    method:            str
    line:              int
    source:            str        # 'static' | 'qualified' | 'soft'
    framework:         str
    in_trycatch:       bool = False
    call_text:         str  = ""
    test_method:       str  = ""
    test_method_start: int  = 0
    test_method_end:   int  = 0


@dataclass
class TestMethod:
    name:         str
    start_line:   int
    end_line:     int
    is_test:      bool = True
    assert_calls: list = field(default_factory=list)


@dataclass
class FileResult:
    path:                   str
    raw_imports:            list[str] = field(default_factory=list)
    frameworks_imported:    set[str]  = field(default_factory=set)
    assert_calls:           list      = field(default_factory=list)
    test_methods:           list      = field(default_factory=list)
    helper_methods:         list[str] = field(default_factory=list)
    has_assert_in_comments: bool      = False
    parse_error:            Optional[str] = None

    @property
    def has_import(self):     return bool(self.frameworks_imported)
    @property
    def has_real_calls(self): return bool(self.assert_calls)
    @property
    def import_only(self):    return self.has_import and not self.has_real_calls
    @property
    def comment_only(self):
        return (
            self.has_assert_in_comments
            and not self.has_real_calls
            and not self.has_import
        )

    def method_counts(self):
        counts = defaultdict(int)
        for c in self.assert_calls:
            counts[c.method] += 1
        return dict(counts)

    def framework_counts(self):
        counts = defaultdict(int)
        for c in self.assert_calls:
            counts[c.framework] += 1
        return dict(counts)


# =============================================================================
# IMPORT ANALYSIS
# =============================================================================

_IMPORT_LINE_RE = re.compile(
    r'^\s*import\s+(?:static\s+)?([a-zA-Z][\w.]*(?:\.\*)?)\s*;', re.MULTILINE
)

def _detect_frameworks_from_imports(raw: str) -> tuple[list[str], set[str]]:
    raw_imports, detected = [], set()
    for m in _IMPORT_LINE_RE.finditer(raw):
        imp = m.group(1)
        raw_imports.append(imp)
        for fw, cfg in FRAMEWORKS.items():
            for pattern in cfg["imports"]:
                if imp == pattern or imp.startswith(pattern.replace(".*", ".")):
                    detected.add(fw)
    return raw_imports, detected


# =============================================================================
# COMMENT SCANNING
# =============================================================================

_COMMENT_RE     = re.compile(r'//[^\n]*|/\*.*?\*/', re.DOTALL)
_ASSERT_WORD_RE = re.compile(r'(?i)\bassert\b')

def _has_assert_in_comments(raw: str) -> bool:
    for m in _COMMENT_RE.finditer(raw):
        if _ASSERT_WORD_RE.search(m.group()):
            return True
    return False


# =============================================================================
# AST HELPERS
# =============================================================================

def _iter_nodes(root):
    """Iteratively yield every node in the subtree (depth-first, pre-order)."""
    stack = [root]
    while stack:
        node = stack.pop()
        yield node
        stack.extend(reversed(node.children))

def _node_text(node, src: bytes) -> str:
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

def _node_start_line(node) -> int:
    return node.start_point[0] + 1

def _node_end_line(node) -> int:
    return node.end_point[0] + 1

def _is_inside_trycatch(node) -> bool:
    cur = node.parent
    while cur:
        if cur.type == "try_statement":
            return True
        cur = cur.parent
    return False

def _method_name_of(node, src: bytes) -> Optional[str]:
    for child in node.children:
        if child.type == "identifier":
            return _node_text(child, src)
    return None

def _object_name_of(node, src: bytes) -> Optional[str]:
    prev = None
    for child in node.children:
        if child.type == ".":
            break
        prev = child
    if prev and prev.type in ("identifier", "type_identifier"):
        return _node_text(prev, src)
    return None


# =============================================================================
# TEST METHOD EXTRACTOR
# =============================================================================

def _extract_test_methods(root_node, src: bytes) -> list[TestMethod]:
    methods = []
    for node in _iter_nodes(root_node):
        if node.type != "method_declaration":
            continue

        is_test, name = False, ""
        for child in node.children:
            if child.type == "modifiers":
                for mod in child.children:
                    if (mod.type == "marker_annotation"
                            and _node_text(mod, src).lstrip("@") == "Test"):
                        is_test = True
            if child.type == "identifier":
                name = _node_text(child, src)

        methods.append(TestMethod(
            name=name,
            start_line=_node_start_line(node),
            end_line=_node_end_line(node),
            is_test=is_test,
        ))
    return methods


def _find_containing_method(line: int, test_methods: list[TestMethod]) -> Optional[TestMethod]:
    for tm in test_methods:
        if tm.start_line <= line <= tm.end_line:
            return tm
    return None


# =============================================================================
# VARIABLE TYPE TRACKER
# =============================================================================

def _collect_local_var_types(root_node, src: bytes) -> dict[str, str]:
    var_types: dict[str, str] = {}

    for node in _iter_nodes(root_node):
        if node.type in ("local_variable_declaration", "field_declaration"):
            type_node = None
            for child in node.children:
                if child.type in ("type_identifier", "generic_type"):
                    type_node = child
                    break
            if type_node:
                type_name = _node_text(type_node, src).split("<")[0].strip()
                for child in node.children:
                    if child.type == "variable_declarator":
                        for sub in child.children:
                            if sub.type == "identifier":
                                var_types[_node_text(sub, src)] = type_name
                                break

        elif node.type == "assignment_expression":
            children = list(node.children)
            if len(children) >= 3:
                lhs = children[0]
                rhs = children[2]
                if lhs.type == "identifier" and rhs.type == "object_creation_expression":
                    for rhs_child in rhs.children:
                        if rhs_child.type in ("type_identifier", "generic_type"):
                            new_type = _node_text(rhs_child, src).split("<")[0].strip()
                            if new_type in ALL_SOFT_CLASSES:
                                var_types[_node_text(lhs, src)] = new_type
                            break

    return var_types


# =============================================================================
# HELPER METHOD DETECTOR
# =============================================================================

def _body_has_assert(method_node, src: bytes) -> bool:
    for node in _iter_nodes(method_node):
        if node.type == "method_invocation":
            name = _method_name_of(node, src)
            if name and name in ALL_ASSERT_METHODS:
                return True
    return False


def _find_helper_method_names(root_node, src: bytes) -> set[str]:
    helpers = set()
    for node in _iter_nodes(root_node):
        if node.type != "method_declaration":
            continue

        is_test = False
        for child in node.children:
            if child.type == "modifiers":
                for mod in child.children:
                    if (mod.type == "marker_annotation"
                            and _node_text(mod, src).lstrip("@") == "Test"):
                        is_test = True

        if not is_test and _body_has_assert(node, src):
            for child in node.children:
                if child.type == "identifier":
                    helpers.add(_node_text(child, src))
                    break
    return helpers


# =============================================================================
# CORE AST WALKER
# =============================================================================

def _collect_assert_calls(
    root_node,
    src: bytes,
    frameworks_imported: set[str],
    var_types: dict[str, str],
    test_methods: list[TestMethod],
) -> list[AssertCall]:

    calls: list[AssertCall] = []

    for node in _iter_nodes(root_node):
        if node.type != "method_invocation":
            continue

        method_name = _method_name_of(node, src)
        obj_name    = _object_name_of(node, src)

        if not method_name:
            continue

        call_text  = _node_text(node, src)
        line       = _node_start_line(node)
        in_try     = _is_inside_trycatch(node)
        containing = _find_containing_method(line, test_methods)
        tm_name    = containing.name       if containing else ""
        tm_start   = containing.start_line if containing else 0
        tm_end     = containing.end_line   if containing else 0
        call       = None

        # A. Static call
        if method_name in ALL_ASSERT_METHODS and obj_name is None:
            fw = _resolve_framework(method_name, frameworks_imported, "static")
            call = AssertCall(
                method=method_name, line=line, source="static", framework=fw,
                in_trycatch=in_try, call_text=call_text[:120],
                test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
            )

        # B. Qualified class call (Assert.assertEquals)
        elif method_name in ALL_ASSERT_METHODS and obj_name in ALL_QUALIFIED_CLASSES:
            fw = _resolve_framework(method_name, frameworks_imported, "qualified", obj_name)
            call = AssertCall(
                method=method_name, line=line, source="qualified", framework=fw,
                in_trycatch=in_try, call_text=call_text[:120],
                test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
            )

        # C. Fully qualified (org.junit.Assert.assertEquals)
        elif method_name in ALL_ASSERT_METHODS and obj_name is not None:
            if re.search(r'org\.(junit|testng)|org\.assertj|org\.hamcrest', call_text):
                fw = _resolve_framework(method_name, frameworks_imported, "fqn")
                call = AssertCall(
                    method=method_name, line=line, source="qualified", framework=fw,
                    in_trycatch=in_try, call_text=call_text[:120],
                    test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
                )

        # D. Soft assertions (softly.assertThat / sa.assertEquals)
        elif obj_name and obj_name in var_types:
            type_name = var_types[obj_name]
            if type_name in ALL_SOFT_CLASSES and method_name in ALL_ASSERT_METHODS:
                fw = SOFT_CLASS_TO_FRAMEWORK.get(type_name, "unknown")
                call = AssertCall(
                    method=method_name, line=line, source="soft", framework=fw,
                    in_trycatch=in_try, call_text=call_text[:120],
                    test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
                )

        if call:
            calls.append(call)
            if containing:
                containing.assert_calls.append(call)

    return calls


def _resolve_framework(
    method: str,
    imported: set[str],
    source: str,
    qualifier: Optional[str] = None,
) -> str:
    candidates = _METHOD_TO_FRAMEWORKS.get(method, [])
    matching   = [fw for fw in candidates if fw in imported]

    if len(matching) == 1:
        return matching[0]
    if len(matching) > 1:
        if qualifier == "Assertions":
            for fw in ("junit5", "assertj"):
                if fw in matching: return fw
        if qualifier == "Assert":
            for fw in ("junit4", "testng"):
                if fw in matching: return fw
        return matching[0]
    if candidates:
        return candidates[0]
    return "unknown"


# =============================================================================
# FILE ANALYZER
# =============================================================================

def analyze_file(filepath: str) -> FileResult:
    result = FileResult(path=filepath)

    try:
        raw_bytes = Path(filepath).read_bytes()
        raw_str   = raw_bytes.decode("utf-8", errors="replace")
    except OSError as e:
        result.parse_error = str(e)
        return result

    result.raw_imports, result.frameworks_imported = \
        _detect_frameworks_from_imports(raw_str)
    result.has_assert_in_comments = _has_assert_in_comments(raw_str)

    try:
        tree = PARSER.parse(raw_bytes)
    except Exception as e:
        result.parse_error = f"tree-sitter parse error: {e}"
        return result

    if tree.root_node.has_error:
        result.parse_error = "syntax warnings (partial parse used)"

    root         = tree.root_node
    var_types    = _collect_local_var_types(root, raw_bytes)
    test_methods = _extract_test_methods(root, raw_bytes)
    helper_names = _find_helper_method_names(root, raw_bytes)

    result.test_methods   = test_methods
    result.helper_methods = sorted(helper_names)
    result.assert_calls   = _collect_assert_calls(
        root, raw_bytes,
        result.frameworks_imported,
        var_types,
        test_methods,
    )
    return result


def analyze_directory(root: str, pattern: str = "*Test*.java") -> list[FileResult]:
    return [analyze_file(str(p)) for p in Path(root).rglob(pattern)]


# =============================================================================
# JSON EXPORT
# =============================================================================

def export_json(
    results: list[FileResult],
    out_path: str = "ast_analysis.json",
    allowed_instances: set[str] | None = None,   # ← NEW
):
    """
    Export per-file analysis to JSON.

    allowed_instances: if provided, only include files whose parent folder
    name (the BBC/BUMP instance dir) is in this set.  Pass None to include
    everything (existing behaviour for compiled_pre / executed_pre).
    """
    output = []
    for r in results:

        # ── Instance filter ────────────────────────────────────────────────
        if allowed_instances is not None:
            instance_name = Path(r.path).parent.name
            # _java_files lives one level deeper: .../<instance>/_java_files/<file>
            if instance_name == "_java_files":
                instance_name = Path(r.path).parent.parent.name
            if instance_name not in allowed_instances:
                continue          # skip files not in the dataset
        # ──────────────────────────────────────────────────────────────────

        test_method_call_ids = set()
        method_summary = []
        for tm in r.test_methods:
            if not tm.is_test:
                continue
            for c in tm.assert_calls:
                test_method_call_ids.add(id(c))
            method_summary.append({
                "name":       tm.name,
                "start_line": tm.start_line,
                "end_line":   tm.end_line,
                "assert_calls": [
                    {
                        "method":      c.method,
                        "line":        c.line,
                        "framework":   c.framework,
                        "in_trycatch": c.in_trycatch,
                        "source":      c.source,
                        "call_text":   c.call_text,
                    }
                    for c in tm.assert_calls
                ],
            })

        flat_calls = []
        for c in r.assert_calls:
            flat_calls.append({
                "method":            c.method,
                "line":              c.line,
                "framework":         c.framework,
                "in_trycatch":       c.in_trycatch,
                "source":            c.source,
                "call_text":         c.call_text,
                "test_method":       c.test_method,
                "test_method_start": c.test_method_start,
                "test_method_end":   c.test_method_end,
                "in_test_method":    id(c) in test_method_call_ids,
            })

        output.append({
            "file":                   Path(r.path).name,
            "path":                   r.path,
            "frameworks_imported":    list(r.frameworks_imported),
            "has_import":             r.has_import,
            "has_real_calls":         r.has_real_calls,
            "import_only":            r.import_only,
            "comment_only":           r.comment_only,
            "has_assert_in_comments": r.has_assert_in_comments,
            "total_assert_calls":     len(r.assert_calls),
            "helper_methods":         r.helper_methods,
            "parse_error":            r.parse_error,
            "assert_calls":           flat_calls,
            "test_methods":           method_summary,
        })

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    print(f"  Details ({len(output)} files): {out_path}")


# =============================================================================
# PREPROCESSING (.txt → .java)
# =============================================================================

def clean_java_code(content: str) -> str:
    content = re.sub(r'^```java\s*\n', '', content, flags=re.MULTILINE)
    content = re.sub(r'^```\s*\n',     '', content, flags=re.MULTILINE)
    content = re.sub(r'\n```\s*$',     '', content, flags=re.MULTILINE)
    content = re.sub(r'^```\s*$',      '', content, flags=re.MULTILINE)
    content = re.sub(r'^.*?(?=\s*(package|import|public|class|//|/\*))',
                     '', content, count=1, flags=re.DOTALL)
    return content.strip()


def extract_class_name(content: str) -> Optional[str]:
    m = re.search(r'public\s+class\s+(\w+)', content)
    return m.group(1) if m else None


def prepare_java_file(txt_path: Path) -> Optional[Path]:
    try:
        content = txt_path.read_text(encoding='utf-8')
    except Exception as e:
        print(f"    ⚠️  Error reading {txt_path.name}: {e}")
        return None

    cleaned    = clean_java_code(content)
    class_name = extract_class_name(cleaned) or txt_path.stem.replace('_prompt', '')

    java_dir  = txt_path.parent / "_java_files"
    java_dir.mkdir(exist_ok=True)
    java_file = java_dir / f"{class_name}.java"

    try:
        java_file.write_text(cleaned, encoding='utf-8')
        return java_file
    except Exception as e:
        print(f"   Error writing {java_file}: {e}")
        return None


def find_test_files(root_dir: str, patterns: list[str]) -> list[Path]:
    files = []
    root  = Path(root_dir)
    for pattern in patterns:
        for f in root.rglob(pattern):
            if "_java_files" not in f.parts:
                files.append(f)
    return sorted(set(files))


# =============================================================================
# STATS + REPORTING
# =============================================================================

def compute_stats(results: list) -> dict:
    total = len(results)

    files_with_calls   = sum(1 for r in results if r.has_real_calls)
    files_import_only  = sum(1 for r in results if r.import_only)
    files_comment_only = sum(1 for r in results if r.comment_only)
    files_no_assert    = sum(
        1 for r in results
        if not r.has_real_calls and not r.import_only and not r.comment_only
    )
    files_with_helpers = sum(1 for r in results if r.helper_methods)

    framework_call_counts: dict[str, int] = defaultdict(int)
    framework_file_counts: dict[str, int] = defaultdict(int)
    method_counts:         dict[str, int] = defaultdict(int)

    for r in results:
        frameworks_seen_in_file = set()
        for c in r.assert_calls:
            framework_call_counts[c.framework] += 1
            method_counts[c.method]            += 1
            frameworks_seen_in_file.add(c.framework)
        for fw in frameworks_seen_in_file:
            framework_file_counts[fw] += 1

    all_test_methods = [
        tm for r in results
        for tm in r.test_methods if tm.is_test
    ]
    total_test_methods = len(all_test_methods)

    test_assert_counts   = [len(tm.assert_calls) for tm in all_test_methods]
    methods_with_asserts = sum(1 for c in test_assert_counts if c > 0)
    methods_no_asserts   = total_test_methods - methods_with_asserts
    total_calls          = sum(test_assert_counts)
    min_asserts          = min(test_assert_counts) if test_assert_counts else 0
    max_asserts          = max(test_assert_counts) if test_assert_counts else 0
    avg_asserts          = total_calls / total_test_methods if total_test_methods > 0 else 0

    in_trycatch = sum(
        sum(1 for c in r.assert_calls if c.in_trycatch) for r in results
    )
    files_with_trycatch = sum(
        1 for r in results if any(c.in_trycatch for c in r.assert_calls)
    )
    soft = sum(1 for r in results if any(c.source == "soft" for c in r.assert_calls))

    return {
        "total_files":                  total,
        "files_with_assert_calls":      files_with_calls,
        "files_import_only":            files_import_only,
        "files_comment_only":           files_comment_only,
        "files_no_assert":              files_no_assert,
        "files_with_helpers":           files_with_helpers,
        "total_test_methods":           total_test_methods,
        "test_methods_with_asserts":    methods_with_asserts,
        "test_methods_without_asserts": methods_no_asserts,
        "total_assert_calls":           total_calls,
        "min_asserts_per_test_method":  min_asserts,
        "max_asserts_per_test_method":  max_asserts,
        "avg_asserts_per_test_method":  avg_asserts,
        "calls_in_trycatch":            in_trycatch,
        "files_with_trycatch_asserts":  files_with_trycatch,
        "files_with_soft_assertions":   soft,
        "framework_call_counts":        dict(framework_call_counts),
        "framework_file_counts":        dict(framework_file_counts),
        "method_breakdown":             dict(method_counts),
    }


def print_summary(stats: dict):
    total = stats['total_files']

    print(f"\n  📊 SUMMARY")
    print(f"  {'-'*66}")

    print(f"\n  FILE LEVEL  (total: {total})")
    print(f"  {'Category':<40} {'Count':>6}  {'%':>6}")
    print(f"  {'-'*55}")
    categories = [
        ("Files with assert calls",  "files_with_assert_calls"),
        ("Files with import only",   "files_import_only"),
        ("Files with comments only", "files_comment_only"),
        ("Files with no assert",     "files_no_assert"),
    ]
    for label, key in categories:
        cnt = stats[key]
        pct = 100 * cnt / total if total > 0 else 0
        print(f"  {label:<40} {cnt:>6}  {pct:>5.1f}%")
    subtotal = sum(stats[k] for _, k in categories)
    print(f"  {'-'*55}")
    print(f"  {'Total':<40} {subtotal:>6}")
    print(f"  Files with helper methods       : {stats['files_with_helpers']}")

    print(f"\n  @TEST METHOD LEVEL")
    print(f"  Total @Test methods             : {stats['total_test_methods']}")
    print(f"  @Test methods with asserts      : {stats['test_methods_with_asserts']}")
    print(f"  @Test methods without asserts   : {stats['test_methods_without_asserts']}")
    print(f"  ")
    print(f"  Total assert calls              : {stats['total_assert_calls']}")
    print(f"  Min asserts per @Test method    : {stats['min_asserts_per_test_method']}")
    print(f"  Max asserts per @Test method    : {stats['max_asserts_per_test_method']}")
    print(f"  Avg asserts per @Test method    : {stats['avg_asserts_per_test_method']:.2f}")
    print(f"  Calls inside try/catch          : {stats['calls_in_trycatch']}")
    print(f"  Files with assert in try/catch  : {stats['files_with_trycatch_asserts']}")
    print(f"  Files with soft assertions      : {stats['files_with_soft_assertions']}")

    if stats['framework_call_counts']:
        print(f"\n  Framework breakdown (calls | files using it):")
        for fw in sorted(stats['framework_call_counts'], key=lambda x: -stats['framework_call_counts'][x]):
            calls = stats['framework_call_counts'].get(fw, 0)
            files = stats['framework_file_counts'].get(fw, 0)
            print(f"    {fw:<20} {calls:>5} calls   {files:>5} files")

    if stats['method_breakdown']:
        print(f"\n  Top 10 assert methods:")
        for m, cnt in sorted(stats['method_breakdown'].items(), key=lambda x: -x[1])[:10]:
            print(f"    {m:<30} {cnt:>5} calls")
    print()


def generate_comparison_report(all_results: dict):
    print(f"\n{'='*70}")
    print(f"  COMPARISON ACROSS DATASETS")
    print(f"{'='*70}\n")

    rows = []
    for name in ["compiled_pre", "executed_pre", "detected_bre"]:
        if name not in all_results or not all_results[name]:
            continue
        s = all_results[name]["stats"]
        rows.append({
            "Dataset":            name,
            "Files":              s["total_files"],
            "w/ Calls":           s["files_with_assert_calls"],
            "Import Only":        s["files_import_only"],
            "No Assert":          s["files_no_assert"],
            "@Test Methods":      s["total_test_methods"],
            "w/ Asserts":         s["test_methods_with_asserts"],
            "w/o Asserts":        s["test_methods_without_asserts"],
            "Total Calls":        s["total_assert_calls"],
            "Min":                s["min_asserts_per_test_method"],
            "Max":                s["max_asserts_per_test_method"],
            "Avg":                f"{s['avg_asserts_per_test_method']:.2f}",
            "In Try/Catch":       s["calls_in_trycatch"],
            "Files w/ Try/Catch": s["files_with_trycatch_asserts"],
        })

    if not rows:
        print("  No data to compare.\n")
        return

    headers = list(rows[0].keys())
    widths  = [
        max(len(str(r.get(h, ""))) for r in rows + [{h: h}])
        for h in headers
    ]
    print("  " + " | ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print("  " + "-+-".join("-" * w for w in widths))
    for row in rows:
        print("  " + " | ".join(str(row.get(h, "")).ljust(w) for h, w in zip(headers, widths)))
    print()


# =============================================================================
# CACHE — save/load all_results stats to survive kernel restarts
# =============================================================================

def save_results(all_results: dict):
    """Save all_results stats to JSON so they survive kernel restarts."""
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    serializable = {
        name: data["stats"]
        for name, data in all_results.items()
        if data
    }
    with open(RESULTS_CACHE, "w") as f:
        json.dump(serializable, f, indent=2)
    print(f"  Results cached: {RESULTS_CACHE}")


def load_results(path: str = RESULTS_CACHE) -> dict:
    """Load cached stats — enough for plotting and comparison reports."""
    with open(path) as f:
        raw = json.load(f)
    return {name: {"stats": stats} for name, stats in raw.items()}


# =============================================================================
# DATASET RUNNER
# =============================================================================

def analyze_dataset(name: str, root_dir: str) -> Optional[dict]:
    print(f"\n{'='*70}")
    print(f"  Analyzing : {name}")
    print(f"  Location  : {root_dir}")
    print(f"{'='*70}\n")

    if not Path(root_dir).exists():
        print(f"    Directory does not exist, skipping.\n")
        return None

    files = find_test_files(root_dir, FILE_PATTERNS)
    if not files:
        print(f"    No test files found.\n")
        return None

    print(f"  Found {len(files)} source files")
    print(f"  Preparing Java files...\n")

    java_files = []
    for f in files:
        if f.suffix == '.txt':
            jf = prepare_java_file(f)
            if jf:
                java_files.append(jf)
        else:
            java_files.append(f)

    if not java_files:
        print(f"    No valid Java files after preprocessing.\n")
        return None

    print(f"  Analyzing {len(java_files)} Java files...\n")

    results = []
    for i, filepath in enumerate(java_files, 1):
        if i % 50 == 0:
            print(f"    Progress: {i}/{len(java_files)}")
        results.append(analyze_file(str(filepath)))

    for r in results:
        if not hasattr(r, 'test_methods') or r.test_methods is None:
            r.test_methods = []

    stats = compute_stats(results)
    stats.update({
        "dataset_name":   name,
        "root_dir":       root_dir,
        "file_count":     len(files),
        "analyzed_count": len(java_files),
    })

    print_summary(stats)
    return {"stats": stats, "results": results}




# =============================================================================
# STEP 1 — Helper: resolve which BBC instances actually exist in a dataset dir
# =============================================================================

def get_present_instances(root_dir: str) -> set[str]:
    """
    Return the set of BBC instance folder names that physically exist
    inside root_dir (e.g. {"BBC07", "BBC12", ...}).
    Only direct subdirectories that match the BBC/BUMP pattern are included.
    """
    import re
    root = Path(root_dir)
    pattern = re.compile(r'^(BBC|BUMP)\d+$', re.IGNORECASE)
    return {
        d.name
        for d in root.iterdir()
        if d.is_dir() and pattern.match(d.name)
    }
    

def save_json_reports(all_results: dict):
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, data in all_results.items():
        if not data:
            continue

        # Save stats JSON (unchanged)
        stats_file = output_dir / f"{name}_stats.json"
        with open(stats_file, "w") as f:
            json.dump(data["stats"], f, indent=2)
        print(f"  Stats  : {stats_file}")

        # For detected_bre, restrict details to instances present in that folder
        allowed: set[str] | None = None
        if name == "detected_bre":
            root_dir = DATASETS[name]
            allowed  = get_present_instances(root_dir)
            print(f"  detected_bre: {len(allowed)} instance folders found → filtering details")

        details_file = str(output_dir / f"{name}_details.json")
        export_json(data["results"], details_file, allowed_instances=allowed)

    print()


# =============================================================================
# MAIN
# =============================================================================

def main():
    print(f"\n{'='*70}")
    print(f"  Java Assert Analysis Runner  (v3 — per-method tracking)")
    print(f"{'='*70}")

    all_results = {}
    for name, path in DATASETS.items():
        result = analyze_dataset(name, path)
        if result:
            all_results[name] = result

    if not all_results:
        print("\n  No datasets analyzed. Check your paths.\n")
        return None

    generate_comparison_report(all_results)
    save_json_reports(all_results)
    save_results(all_results)

    print(f"\n{'='*70}")
    print(f"   Analysis complete!")
    print(f"   Reports saved to : {OUTPUT_DIR}")
    print(f"   Cleaned .java    : each instance/_java_files/")
    print(f"{'='*70}\n")

    return all_results


# =============================================================================
# ENTRY POINT
# =============================================================================

# In a notebook cell, call:
#   all_results = main()
#
# After a kernel restart, reload without re-running analysis:
#   all_results = load_results()

if __name__ == "__main__":
    all_results = main()


  Java Assert Analysis Runner  (v3 — per-method tracking)

  Analyzing : compiled_pre
  Location  : /Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre

  Found 971 source files
  Preparing Java files...

  Analyzing 971 Java files...

    Progress: 50/971
    Progress: 100/971
    Progress: 150/971
    Progress: 200/971
    Progress: 250/971
    Progress: 300/971
    Progress: 350/971
    Progress: 400/971
    Progress: 450/971
    Progress: 500/971
    Progress: 550/971
    Progress: 600/971
    Progress: 650/971
    Progress: 700/971
    Progress: 750/971
    Progress: 800/971
    Progress: 850/971
    Progress: 900/971
    Progress: 950/971

  📊 SUMMARY
  ------------------------------------------------------------------

  FILE LEVEL  (total: 971)
  Category                                  Count       %
  -------------------------------------------------------
  Files with assert calls                     904   93.1%
  Files with import only                       

Step 2: This is Qualitative analysis, In the previous step using Exp3AssertionCount.ipynb, we saw that LLMs freqwently provided Assert() however manually looking at a few logs showed that they fail in initialization step rather than reaching the assert() calls. Therefor I want to explore, out of all , how many failed due to assert calls.

In [ ]:
"""
Log Parser — Assertion Execution Rate Analyzer (v2)

Two separate analyses + one combined report:

  Analysis 1 (has_real_calls=True):
    Did the assert() actually execute during test run, or did the test
    fail before reaching it? How often, and what was the failure type?

  Analysis 2 (has_real_calls=False):
    No assertions in the file — what caused the failure?
    Unknown failure types go into "other" for manual investigation.

Input:
  - LOGS_DIR   : folder of BBC06_BBC06U155Test.java_breaking_single.log files
  - JSON_PATH  : detected_bre_details.json (from AssertionCount / AssertAnalyzer)

Output:
  - execution_rate_summary.json
  - execution_rate_details.csv         (Analysis 1 — per @Test method rows)
  - no_assert_failure_breakdown.csv    (Analysis 2 — per file rows)
"""

import re
import csv
import json
import sys
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional


# =============================================================================
# CONFIGURATION
# =============================================================================

LOGS_DIR   = "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs"
JSON_PATH  = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/detected_bre_details.json"
OUTPUT_DIR = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/Assert_log_analysis"


# =============================================================================
# STAGE / FAILURE KIND HELPERS
# =============================================================================

_KNOWN_STAGES = ["static_init", "constructor", "test_method", "unknown", "other"]

_STAGE_LABELS = {
    "static_init": "Static initialization (<clinit>)",
    "constructor":  "Constructor (<init>)",
    "test_method":  "Inside @Test method body",
    "unknown":      "Unknown (line found, unclassified)",
    "other":        "Other (no stack line — manual investigation)",
}


def _ordered_stages(kind_map: dict) -> list[str]:
    extra = sorted(k for k in kind_map if k not in _KNOWN_STAGES)
    return _KNOWN_STAGES + extra


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class LogResult:
    """Parsed result for one log file."""
    log_file:   str
    instance:   str          # e.g. "BBC06"
    test_file:  str          # e.g. "BBC06U155Test.java"
    class_name: str          # e.g. "BBC06U155Test"

    # From log
    failure_line:      Optional[int] = None
    failure_kind:      str = "other"   # static_init | constructor | test_method | unknown | other
    failure_exception: str = ""
    stack_trace:       str = ""

    # From AST JSON (filled during merge)
    has_real_calls:        bool = False
    total_asserts_in_file: int  = 0
    assert_calls:          list = field(default_factory=list)   # flat list
    test_methods:          list = field(default_factory=list)   # list of method dicts from JSON

    # Computed (Analysis 1 only — filled when has_real_calls=True)
    method_results: list = field(default_factory=list)


@dataclass
class MethodExecutionResult:
    """
    Analysis 1 — execution analysis for one @Test method inside a file
    that has assert calls.
    """
    method_name:         str
    method_start:        int
    method_end:          int
    total_asserts:       int
    asserts_before_fail: int   # upper bound: could have executed
    asserts_after_fail:  int   # definitely did NOT execute
    assert_executed:     bool  # True if asserts_before_fail > 0
    failure_line:        int
    failure_kind:        str
    failure_in_method:   bool  # failure_line falls inside this method's range


# =============================================================================
# LOG FILENAME PARSER
# =============================================================================

def parse_log_filename(filename: str) -> tuple[str, str]:
    """
    BBC06_BBC06U155Test.java_breaking_single.log
      → instance  = "BBC06"
      → test_file = "BBC06U155Test.java"
    """
    name  = filename.replace("_breaking_single.log", "")
    parts = name.split("_", 1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return name, name


# =============================================================================
# LOG CONTENT PARSER
# =============================================================================

_STACK_LINE_RE = re.compile(
    r'at\s+([\w$.]+)\.([\w<>$]+)\((\w+\.java):(\d+)\)'
)
_EXCEPTION_RE = re.compile(
    r'^(java\.[\w.]+(?:Error|Exception)|'
    r'org\.junit\.[\w.]+(?:Error|Exception)|'
    r'org\.opentest4j\.[\w.]+(?:Error|Exception))[:\s]',
    re.MULTILINE
)


def _classify_failure_kind(method_name: str) -> str:
    if "<clinit>" in method_name:
        return "static_init"
    if "<init>" in method_name:
        return "constructor"
    if method_name.startswith("test"):
        return "test_method"
    return "unknown"


def parse_log_file(log_path: Path) -> LogResult:
    instance, test_file = parse_log_filename(log_path.name)
    class_name = test_file.replace(".java", "")

    result = LogResult(
        log_file=log_path.name,
        instance=instance,
        test_file=test_file,
        class_name=class_name,
    )

    try:
        text = log_path.read_text(encoding="utf-8", errors="replace")
    except OSError as e:
        result.failure_exception = f"Cannot read log: {e}"
        result.failure_kind = "other"
        return result

    # Primary exception
    exc_match = _EXCEPTION_RE.search(text)
    if exc_match:
        line_start = text.rfind('\n', 0, exc_match.start()) + 1
        line_end   = text.find('\n', exc_match.start())
        result.failure_exception = text[line_start:line_end].strip()[:200]

    # Stack trace — first frame referencing the test file → failure_line
    for m in _STACK_LINE_RE.finditer(text):
        fqn       = m.group(1)
        method    = m.group(2)
        java_file = m.group(3)
        line_no   = int(m.group(4))

        if java_file == test_file or fqn.endswith(class_name):
            result.failure_line = line_no
            result.failure_kind = _classify_failure_kind(method)
            break

    # If no stack frame matched our test file, keep "other"
    if result.failure_line is None:
        result.failure_kind = "other"

    # Short stack snippet
    lines = []
    for line in text.split('\n'):
        s = line.strip()
        if s.startswith('at ') or s.startswith('Caused by:'):
            lines.append(s)
            if len(lines) >= 5:
                break
    result.stack_trace = '\n'.join(lines)

    return result


# =============================================================================
# AST JSON LOADER
# =============================================================================

def load_ast_json(json_path: str) -> dict[str, dict]:
    """
    Returns a dict keyed by filename (e.g. "BBC06U155Test.java") → entry dict.
    """
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    return {Path(entry["file"]).name: entry for entry in data}


# =============================================================================
# MERGE: attach AST info to each LogResult
# =============================================================================

def merge(log_results: list[LogResult], ast_lookup: dict[str, dict]) -> list[LogResult]:
    matched, unmatched = 0, []

    for lr in log_results:
        ast = ast_lookup.get(lr.test_file)
        if ast is None:
            unmatched.append(lr.test_file)
            continue
        matched += 1
        lr.has_real_calls        = ast.get("has_real_calls", False)
        lr.total_asserts_in_file = ast.get("total_assert_calls", 0)
        lr.assert_calls          = ast.get("assert_calls", [])
        lr.test_methods          = ast.get("test_methods", [])

        # Analysis 1 — compute per-method execution only when asserts exist
        if lr.has_real_calls:
            lr.method_results = _compute_method_execution(lr)

    if unmatched:
        print(f"  ⚠  {len(unmatched)} logs had no AST match (excluded from analysis): {unmatched[:5]}")
    print(f"  ✓  Matched {matched}/{len(log_results)} logs to AST entries")

    # Return ONLY matched results — unmatched have no AST data and must not
    # pollute either analysis or any counts.
    matched_results = [lr for lr in log_results if lr.test_file not in set(unmatched)]
    return matched_results


def _compute_method_execution(lr: LogResult) -> list[MethodExecutionResult]:
    results      = []
    failure_line = lr.failure_line

    for tm in lr.test_methods:
        m_start = tm["start_line"]
        m_end   = tm["end_line"]
        calls   = tm.get("assert_calls", [])
        total   = len(calls)

        if failure_line is None:
            before    = 0
            after     = total
            in_method = False
        else:
            before    = sum(1 for c in calls if c["line"] < failure_line)
            after     = sum(1 for c in calls if c["line"] >= failure_line)
            in_method = m_start <= failure_line <= m_end

        results.append(MethodExecutionResult(
            method_name=tm["name"],
            method_start=m_start,
            method_end=m_end,
            total_asserts=total,
            asserts_before_fail=before,
            asserts_after_fail=after,
            assert_executed=before > 0,
            failure_line=failure_line or 0,
            failure_kind=lr.failure_kind,
            failure_in_method=in_method,
        ))
    return results


# =============================================================================
# ANALYSIS 1 — has_real_calls = True
# =============================================================================

def analysis1_assert_execution(log_results: list[LogResult]) -> dict:
    """
    For files with has_real_calls=True:
    Did the assertions execute, or did the test fail before reaching them?
    """
    subset = [lr for lr in log_results if lr.has_real_calls]

    total_files   = len(subset)
    total_methods = sum(len(lr.method_results) for lr in subset)
    total_asserts = sum(mr.total_asserts       for lr in subset for mr in lr.method_results)
    before_fail   = sum(mr.asserts_before_fail for lr in subset for mr in lr.method_results)
    after_fail    = sum(mr.asserts_after_fail  for lr in subset for mr in lr.method_results)
    exec_rate     = before_fail / total_asserts * 100 if total_asserts > 0 else 0

    files_assert_executed = sum(
        1 for lr in subset if any(mr.assert_executed for mr in lr.method_results)
    )
    files_failed_before   = total_files - files_assert_executed

    # Failure kind breakdown
    kind_counts: dict[str, int] = defaultdict(int)
    for lr in subset:
        kind_counts[lr.failure_kind] += 1

    # Per-stage execution rate
    stage_stats: dict[str, dict] = defaultdict(lambda: {"total": 0, "before": 0})
    for lr in subset:
        for mr in lr.method_results:
            s = stage_stats[mr.failure_kind]
            s["total"]  += mr.total_asserts
            s["before"] += mr.asserts_before_fail

    per_stage = {}
    for kind, s in stage_stats.items():
        per_stage[kind] = {
            "total_asserts":         s["total"],
            "asserts_before_fail":   s["before"],
            "upper_bound_exec_rate": round(s["before"] / s["total"] * 100, 2) if s["total"] > 0 else 0,
        }

    # Exception category breakdown (same as Analysis 2)
    exc_categories: dict[str, int] = defaultdict(int)
    other_files: list[str] = []
    for lr in subset:
        cat = _categorize_exception(lr.failure_exception)
        exc_categories[cat] += 1
        if cat == "other":
            other_files.append(f"{lr.instance}/{lr.test_file}")

    return {
        "total_files":                    total_files,
        "total_test_methods":             total_methods,
        "total_assert_calls":             total_asserts,
        "asserts_before_fail":            before_fail,
        "asserts_after_fail":             after_fail,
        "upper_bound_exec_rate_pct":      round(exec_rate, 2),
        "files_where_assert_executed":    files_assert_executed,
        "files_failed_before_assert":     files_failed_before,
        "failure_kind_breakdown":         dict(kind_counts),
        "per_stage_exec_rates":           per_stage,
        "exception_category_breakdown":   dict(exc_categories),
        "other_files_for_investigation":  other_files,
        "_note": "asserts_before_fail is an UPPER BOUND — actual may be lower due to branches",
    }


# =============================================================================
# ANALYSIS 2 — has_real_calls = False
# =============================================================================

# Known exception → meaningful category mapping
_EXCEPTION_CATEGORIES = {
    "NullPointerException":        "null_pointer",
    "ClassNotFoundException":      "class_not_found",
    "NoClassDefFoundError":        "class_not_found",
    "NoSuchMethodError":           "missing_method",
    "NoSuchMethodException":       "missing_method",
    "NoSuchFieldError":            "missing_field",
    "AbstractMethodError":         "missing_method",
    "IncompatibleClassChangeError":"incompatible_class",
    "ClassCastException":          "class_cast",
    "IllegalAccessError":          "illegal_access",
    "IllegalArgumentException":    "illegal_argument",
    "VerifyError":                 "verify_error",
    "LinkageError":                "linkage_error",
    "ExceptionInInitializerError": "static_init_error",
    "AssertionError":              "assertion_error",
    "ComparisonFailure":           "assertion_error",
}


def _categorize_exception(exception_str: str) -> str:
    """Map a raw exception string to a known category, or 'other'."""
    if not exception_str:
        return "other"
    short = exception_str.split(":")[0].strip().split(".")[-1]
    return _EXCEPTION_CATEGORIES.get(short, "other")


def analysis2_no_assert_failures(log_results: list[LogResult]) -> dict:
    """
    For files with has_real_calls=False:
    What caused the failure? 'other' = unknown, needs manual investigation.
    """
    subset = [lr for lr in log_results if not lr.has_real_calls]

    total_files = len(subset)

    # Failure kind (lifecycle stage)
    stage_counts: dict[str, int] = defaultdict(int)
    for lr in subset:
        stage_counts[lr.failure_kind] += 1

    # Exception category
    exc_categories: dict[str, int] = defaultdict(int)
    other_files: list[str] = []
    for lr in subset:
        cat = _categorize_exception(lr.failure_exception)
        exc_categories[cat] += 1
        if cat == "other":
            other_files.append(f"{lr.instance}/{lr.test_file}")

    return {
        "total_files":              total_files,
        "failure_stage_breakdown":  dict(stage_counts),
        "exception_category_breakdown": dict(exc_categories),
        "other_files_for_investigation": other_files,
        "_note": "'other' category = unrecognized exception, listed above for manual review",
    }


# =============================================================================
# COMBINED CONSOLE REPORT
# =============================================================================

def print_report(a1: dict, a2: dict):
    W = 70
    sep = "=" * W
    thin = "-" * 66

    # ── Analysis 1 ────────────────────────────────────────────────────────────
    print(f"\n{sep}")
    print(f"  ANALYSIS 1 — Files WITH assertions (has_real_calls=True)")
    print(f"{sep}")
    print(f"  Files analyzed                  : {a1['total_files']}")
    print(f"  @Test methods                   : {a1['total_test_methods']}")
    print(f"  Total assert calls in code      : {a1['total_assert_calls']}")
    print()
    print(f"  Files where assert DID execute  : {a1['files_where_assert_executed']}")
    print(f"  Files failed BEFORE any assert  : {a1['files_failed_before_assert']}")
    print()
    print(f"  Assertions before failure line  : {a1['asserts_before_fail']}  "
          f"({a1['upper_bound_exec_rate_pct']:.1f}%)  ← upper bound")
    print(f"  Assertions after  failure line  : {a1['asserts_after_fail']}")
    print(f"  ⚠  upper bound: branches may mean fewer actually ran")
    print()

    print(f"  Failure lifecycle stage:")
    print(f"  {thin}")
    for kind in _ordered_stages(a1["failure_kind_breakdown"]):
        cnt   = a1["failure_kind_breakdown"].get(kind, 0)
        total = a1["total_files"]
        pct   = cnt / total * 100 if total > 0 else 0
        label = _STAGE_LABELS.get(kind, kind)
        print(f"  {label:<45} {cnt:>4}  ({pct:.1f}%)")
    print()

    print(f"  Execution rate by failure stage:")
    print(f"  {thin}")
    for kind in _ordered_stages(a1["per_stage_exec_rates"]):
        s     = a1["per_stage_exec_rates"].get(kind)
        if not s:
            continue
        label = _STAGE_LABELS.get(kind, kind)
        print(f"  {label:<45} {s['upper_bound_exec_rate']:>5.1f}%  "
              f"({s['asserts_before_fail']}/{s['total_asserts']} asserts)")
    print()

    print(f"  Exception category breakdown:")
    print(f"  {thin}")
    total1 = a1["total_files"]
    for cat, cnt in sorted(a1["exception_category_breakdown"].items(), key=lambda x: -x[1]):
        pct = cnt / total1 * 100 if total1 > 0 else 0
        print(f"  {cat:<45} {cnt:>4}  ({pct:.1f}%)")

    other_count = a1["exception_category_breakdown"].get("other", 0)
    if other_count > 0:
        print(f"\n  ⚠  {other_count} file(s) in 'other' — needs manual investigation:")
        for f in a1["other_files_for_investigation"][:10]:
            print(f"       {f}")
        if len(a1["other_files_for_investigation"]) > 10:
            print(f"       ... and {len(a1['other_files_for_investigation']) - 10} more (see JSON)")

    # ── Analysis 2 ────────────────────────────────────────────────────────────
    print(f"\n{sep}")
    print(f"  ANALYSIS 2 — Files WITHOUT assertions (has_real_calls=False)")
    print(f"{sep}")
    print(f"  Files analyzed                  : {a2['total_files']}")
    print()

    print(f"  Failure lifecycle stage:")
    print(f"  {thin}")
    total2 = a2["total_files"]
    for kind in _ordered_stages(a2["failure_stage_breakdown"]):
        cnt   = a2["failure_stage_breakdown"].get(kind, 0)
        pct   = cnt / total2 * 100 if total2 > 0 else 0
        label = _STAGE_LABELS.get(kind, kind)
        print(f"  {label:<45} {cnt:>4}  ({pct:.1f}%)")
    print()

    print(f"  Exception category breakdown:")
    print(f"  {thin}")
    for cat, cnt in sorted(a2["exception_category_breakdown"].items(), key=lambda x: -x[1]):
        pct   = cnt / total2 * 100 if total2 > 0 else 0
        print(f"  {cat:<45} {cnt:>4}  ({pct:.1f}%)")

    other_count = a2["exception_category_breakdown"].get("other", 0)
    if other_count > 0:
        print(f"\n  ⚠  {other_count} file(s) in 'other' — needs manual investigation:")
        for f in a2["other_files_for_investigation"][:10]:
            print(f"       {f}")
        if len(a2["other_files_for_investigation"]) > 10:
            print(f"       ... and {len(a2['other_files_for_investigation']) - 10} more (see JSON)")

    print(f"\n{sep}\n")


# =============================================================================
# EXPORT FUNCTIONS
# =============================================================================

def export_analysis1_csv(log_results: list[LogResult], out_path: str):
    """Per-@Test-method rows for files with assertions."""
    subset = [lr for lr in log_results if lr.has_real_calls]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "instance", "test_file", "test_method",
            "method_start", "method_end",
            "total_asserts", "asserts_before_fail", "asserts_after_fail",
            "assert_executed", "upper_bound_exec_rate_pct",
            "failure_kind", "failure_line", "failure_in_method",
            "exception_category", "failure_exception",
        ])
        for lr in subset:
            for mr in lr.method_results:
                rate = (
                    mr.asserts_before_fail / mr.total_asserts * 100
                    if mr.total_asserts > 0 else 0.0
                )
                w.writerow([
                    lr.instance, lr.test_file, mr.method_name,
                    mr.method_start, mr.method_end,
                    mr.total_asserts, mr.asserts_before_fail, mr.asserts_after_fail,
                    mr.assert_executed, f"{rate:.1f}",
                    mr.failure_kind, mr.failure_line, mr.failure_in_method,
                    _categorize_exception(lr.failure_exception),
                    lr.failure_exception[:120],
                ])
    print(f"  CSV (Analysis 1) → {out_path}")


def export_analysis2_csv(log_results: list[LogResult], out_path: str):
    """Per-file rows for files without assertions."""
    subset = [lr for lr in log_results if not lr.has_real_calls]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "instance", "test_file",
            "failure_kind", "failure_line",
            "exception_category", "failure_exception",
        ])
        for lr in subset:
            w.writerow([
                lr.instance, lr.test_file,
                lr.failure_kind, lr.failure_line or "",
                _categorize_exception(lr.failure_exception),
                lr.failure_exception[:120],
            ])
    print(f"  CSV (Analysis 2) → {out_path}")


def export_summary_json(a1: dict, a2: dict, out_path: str):
    summary = {
        "analysis_1_has_real_calls_true":  a1,
        "analysis_2_has_real_calls_false": a2,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print(f"  JSON summary      → {out_path}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  Log Parser — Assertion Execution Rate Analyzer (v2)")
    print(f"{'='*70}")
    print(f"  Logs dir   : {LOGS_DIR}")
    print(f"  AST JSON   : {JSON_PATH}")
    print(f"  Output dir : {output_dir}")

    # ── Load ──────────────────────────────────────────────────────────────────
    log_files = sorted(Path(LOGS_DIR).glob("*_breaking_single.log"))
    if not log_files:
        print("  No *_breaking_single.log files found.")
        sys.exit(1)

    print(f"\n  Found {len(log_files)} log files. Parsing...")
    log_results = [parse_log_file(f) for f in log_files]

    print(f"  Loading AST JSON...")
    ast_lookup  = load_ast_json(JSON_PATH)

    log_results = merge(log_results, ast_lookup)

    # ── Analyse ───────────────────────────────────────────────────────────────
    a1 = analysis1_assert_execution(log_results)
    a2 = analysis2_no_assert_failures(log_results)

    # ── Report + Export ───────────────────────────────────────────────────────
    print_report(a1, a2)

    export_analysis1_csv(log_results, str(output_dir / "execution_rate_details.csv"))
    export_analysis2_csv(log_results, str(output_dir / "no_assert_failure_breakdown.csv"))
    export_summary_json(a1, a2,       str(output_dir / "execution_rate_summary.json"))

    print(f"\n{'='*70}")
    print(f"  Done!")
    print(f"{'='*70}\n")


if __name__ == "__main__":
    main()


  Log Parser — Assertion Execution Rate Analyzer (v2)
  Logs dir   : /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs
  AST JSON   : /Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/detected_bre_details.json
  Output dir : /Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/log_analysis

  Found 440 log files. Parsing...
  Loading AST JSON...
  ⚠  272 logs had no AST match (excluded from analysis): ['BBC04U16Test.java', 'BBC04U17Test.java', 'BBC04U21Test.java', 'BBC04U3Test.java', 'BBC06U122Test.java']
  ✓  Matched 168/440 logs to AST entries

  ANALYSIS 1 — Files WITH assertions (has_real_calls=True)
  Files analyzed                  : 135
  @Test methods                   : 139
  Total assert calls in code      : 241

  Files where assert DID execute  : 3
  Files failed BEFORE any assert  : 132

  Assertions before failure line  : 22  (9.1%)  ← upper bound
  Assertions after  failure line  : 219
  ⚠  upper bound: branches may mean fewer actually ran

  Fa

Adding trycatch block level granularity whereever assert calls are present to understand antipatterns.

In [15]:
"""
Log Parser — Assertion Execution Rate Analyzer (v3)

Two separate analyses + one combined report:

  Analysis 1 (has_real_calls=True):
    Did the assert() actually execute during test run, or did the test
    fail before reaching it? How often, and what was the failure type?

  Analysis 2 (has_real_calls=False):
    No assertions in the file — what caused the failure?
    Unknown failure types go into "other" for manual investigation.

Input:
  - LOGS_DIR  : folder of BBC06_BBC06U155Test.java_breaking_single.log files
  - JSON_PATH : detected_bre_details.json (from AssertionCount / AssertAnalyzer)

Output:
  - execution_rate_summary.json
  - execution_rate_details.csv      (Analysis 1 — per @Test method rows)
  - no_assert_failure_breakdown.csv (Analysis 2 — per file rows)
"""

import re
import csv
import json
import sys
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional


# =============================================================================
# CONFIGURATION
# =============================================================================

LOGS_DIR   = "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs"
JSON_PATH  = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/detected_bre_details.json"
OUTPUT_DIR = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/Assert_log_analysis"


# =============================================================================
# STAGE / FAILURE KIND HELPERS
# =============================================================================

_KNOWN_STAGES = ["static_init", "constructor", "test_method", "unknown", "other"]

_STAGE_LABELS = {
    "static_init": "Static initialization (<clinit>)",
    "constructor":  "Constructor (<init>)",
    "test_method":  "Inside @Test method body",
    "unknown":      "Unknown (line found, unclassified)",
    "other":        "Other (no stack line — manual investigation)",
}


def _ordered_stages(kind_map: dict) -> list[str]:
    extra = sorted(k for k in kind_map if k not in _KNOWN_STAGES)
    return _KNOWN_STAGES + extra


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class LogResult:
    """Parsed result for one log file."""
    log_file:   str
    instance:   str       # e.g. "BBC06"
    test_file:  str       # e.g. "BBC06U155Test.java"
    class_name: str       # e.g. "BBC06U155Test"

    # From log
    failure_line:      Optional[int] = None
    failure_kind:      str = "other"  # static_init | constructor | test_method | unknown | other
    failure_exception: str = ""
    stack_trace:       str = ""

    # From AST JSON (filled during merge)
    has_real_calls:        bool = False
    total_asserts_in_file: int  = 0
    assert_calls:          list = field(default_factory=list)  # flat list from JSON
    test_methods:          list = field(default_factory=list)  # test_methods[] from JSON

    # Computed (Analysis 1 only)
    method_results: list = field(default_factory=list)


@dataclass
class MethodExecutionResult:
    """Analysis 1 — execution analysis for one @Test method."""
    method_name:          str
    method_start:         int
    method_end:           int
    total_asserts:        int
    asserts_before_fail:  int   # upper bound: assert line < failure line
    asserts_after_fail:   int   # definitely did NOT execute
    assert_executed:      bool  # True if asserts_before_fail > 0
    failure_line:         int
    failure_kind:         str
    failure_in_method:    bool  # failure_line falls inside this method's range
    # try/catch sub-counts scoped to this method
    trycatch_total:           int = 0
    trycatch_before_fail:     int = 0
    trycatch_after_fail:      int = 0
    non_trycatch_total:       int = 0
    non_trycatch_before_fail: int = 0
    non_trycatch_after_fail:  int = 0


# =============================================================================
# LOG FILENAME PARSER
# =============================================================================

def parse_log_filename(filename: str) -> tuple[str, str]:
    """
    BBC06_BBC06U155Test.java_breaking_single.log
      → instance  = "BBC06"
      → test_file = "BBC06U155Test.java"
    """
    name  = filename.replace("_breaking_single.log", "")
    parts = name.split("_", 1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return name, name


# =============================================================================
# LOG CONTENT PARSER
# =============================================================================

_STACK_LINE_RE = re.compile(
    r'at\s+([\w$.]+)\.([\w<>$]+)\((\w+\.java):(\d+)\)'
)
_EXCEPTION_RE = re.compile(
    r'^(java\.[\w.]+(?:Error|Exception)|'
    r'org\.junit\.[\w.]+(?:Error|Exception)|'
    r'org\.opentest4j\.[\w.]+(?:Error|Exception))[:\s]',
    re.MULTILINE
)


def _classify_failure_kind(method_name: str) -> str:
    if "<clinit>" in method_name: return "static_init"
    if "<init>"   in method_name: return "constructor"
    if method_name.startswith("test"): return "test_method"
    return "unknown"


def parse_log_file(log_path: Path) -> LogResult:
    instance, test_file = parse_log_filename(log_path.name)
    result = LogResult(
        log_file=log_path.name,
        instance=instance,
        test_file=test_file,
        class_name=test_file.replace(".java", ""),
    )

    try:
        text = log_path.read_text(encoding="utf-8", errors="replace")
    except OSError as e:
        result.failure_exception = f"Cannot read log: {e}"
        result.failure_kind = "other"
        return result

    # Primary exception
    exc_match = _EXCEPTION_RE.search(text)
    if exc_match:
        line_start = text.rfind('\n', 0, exc_match.start()) + 1
        line_end   = text.find('\n', exc_match.start())
        result.failure_exception = text[line_start:line_end].strip()[:200]

    # First stack frame referencing the test file → failure_line + kind
    for m in _STACK_LINE_RE.finditer(text):
        fqn, method, java_file, line_no = m.group(1), m.group(2), m.group(3), int(m.group(4))
        if java_file == test_file or fqn.endswith(result.class_name):
            result.failure_line = line_no
            result.failure_kind = _classify_failure_kind(method)
            break

    # No matching stack frame — keep "other"
    if result.failure_line is None:
        result.failure_kind = "other"

    # Short stack snippet for reference
    lines = []
    for line in text.split('\n'):
        s = line.strip()
        if s.startswith('at ') or s.startswith('Caused by:'):
            lines.append(s)
            if len(lines) >= 5:
                break
    result.stack_trace = '\n'.join(lines)
    return result


# =============================================================================
# EXCEPTION CATEGORIZER
# =============================================================================

_EXCEPTION_CATEGORIES = {
    "NullPointerException":         "null_pointer",
    "ClassNotFoundException":       "class_not_found",
    "NoClassDefFoundError":         "class_not_found",
    "NoSuchMethodError":            "missing_method",
    "NoSuchMethodException":        "missing_method",
    "NoSuchFieldError":             "missing_field",
    "AbstractMethodError":          "missing_method",
    "IncompatibleClassChangeError": "incompatible_class",
    "ClassCastException":           "class_cast",
    "IllegalAccessError":           "illegal_access",
    "IllegalArgumentException":     "illegal_argument",
    "VerifyError":                  "verify_error",
    "LinkageError":                 "linkage_error",
    "ExceptionInInitializerError":  "static_init_error",
    "AssertionError":               "assertion_error",
    "ComparisonFailure":            "assertion_error",
}


def _categorize_exception(exception_str: str) -> str:
    """Map a raw exception string to a known category, or 'other'."""
    if not exception_str:
        return "other"
    short = exception_str.split(":")[0].strip().split(".")[-1]
    return _EXCEPTION_CATEGORIES.get(short, "other")


# =============================================================================
# AST JSON LOADER
# =============================================================================

def load_ast_json(json_path: str) -> dict[str, dict]:
    """Returns dict keyed by filename (e.g. 'BBC06U155Test.java') → entry."""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    return {Path(entry["file"]).name: entry for entry in data}


# =============================================================================
# MERGE: attach AST info to each LogResult
# =============================================================================

def merge(log_results: list[LogResult], ast_lookup: dict[str, dict]) -> list[LogResult]:
    matched, unmatched = 0, []

    for lr in log_results:
        ast = ast_lookup.get(lr.test_file)
        if ast is None:
            unmatched.append(lr.test_file)
            continue
        matched += 1
        lr.has_real_calls        = ast.get("has_real_calls", False)
        lr.total_asserts_in_file = ast.get("total_assert_calls", 0)
        lr.assert_calls          = ast.get("assert_calls", [])
        lr.test_methods          = ast.get("test_methods", [])
        if lr.has_real_calls:
            lr.method_results = _compute_method_execution(lr)

    if unmatched:
        print(f"  ⚠  {len(unmatched)} logs had no AST match (excluded): {unmatched[:5]}")
    print(f"  ✓  Matched {matched}/{len(log_results)} logs to AST entries")

    unmatched_set = set(unmatched)
    return [lr for lr in log_results if lr.test_file not in unmatched_set]


def _compute_method_execution(lr: LogResult) -> list[MethodExecutionResult]:
    results      = []
    failure_line = lr.failure_line

    for tm in lr.test_methods:
        m_start = tm["start_line"]
        m_end   = tm["end_line"]
        calls   = tm.get("assert_calls", [])
        tc      = [c for c in calls if     c.get("in_trycatch", False)]
        non_tc  = [c for c in calls if not c.get("in_trycatch", False)]

        if failure_line is None:
            before = after = 0
            after  = len(calls)
            in_method      = False
            tc_before,     tc_after     = 0, len(tc)
            non_tc_before, non_tc_after = 0, len(non_tc)
        else:
            before    = sum(1 for c in calls   if c["line"] < failure_line)
            after     = sum(1 for c in calls   if c["line"] >= failure_line)
            in_method = m_start <= failure_line <= m_end
            tc_before     = sum(1 for c in tc     if c["line"] < failure_line)
            tc_after      = sum(1 for c in tc     if c["line"] >= failure_line)
            non_tc_before = sum(1 for c in non_tc if c["line"] < failure_line)
            non_tc_after  = sum(1 for c in non_tc if c["line"] >= failure_line)

        results.append(MethodExecutionResult(
            method_name=tm["name"],
            method_start=m_start,
            method_end=m_end,
            total_asserts=len(calls),
            asserts_before_fail=before,
            asserts_after_fail=after,
            assert_executed=before > 0,
            failure_line=failure_line or 0,
            failure_kind=lr.failure_kind,
            failure_in_method=in_method,
            trycatch_total=len(tc),
            trycatch_before_fail=tc_before,
            trycatch_after_fail=tc_after,
            non_trycatch_total=len(non_tc),
            non_trycatch_before_fail=non_tc_before,
            non_trycatch_after_fail=non_tc_after,
        ))
    return results


# =============================================================================
# ANALYSIS 1 — has_real_calls = True
# =============================================================================

def _file_tc_stats(lrs: list[LogResult]) -> dict:
    """
    File-level try/catch assert counts for a given subset of LogResults.
    Reads from the flat assert_calls[] — independent of @Test method boundaries.

    before_fail = assert line < failure line  (upper bound — may not have run)
    after_fail  = assert line >= failure line (definitely did not run)
    """
    tc_total = ntc_total = tc_before = ntc_before = 0
    files_with_tc = files_with_ntc = 0
    for lr in lrs:
        f_tc  = [c for c in lr.assert_calls if     c.get("in_trycatch", False)]
        f_ntc = [c for c in lr.assert_calls if not c.get("in_trycatch", False)]
        tc_total  += len(f_tc)
        ntc_total += len(f_ntc)
        tc_before  += sum(1 for c in f_tc  if lr.failure_line and c["line"] < lr.failure_line)
        ntc_before += sum(1 for c in f_ntc if lr.failure_line and c["line"] < lr.failure_line)
        if f_tc:  files_with_tc  += 1
        if f_ntc: files_with_ntc += 1
    return {
        "files_with_assert_in_trycatch":     files_with_tc,
        "files_with_assert_outside_trycatch": files_with_ntc,
        "in_trycatch": {
            "total":       tc_total,
            "before_fail": tc_before,
            "after_fail":  tc_total - tc_before,
        },
        "not_in_trycatch": {
            "total":       ntc_total,
            "before_fail": ntc_before,
            "after_fail":  ntc_total - ntc_before,
        },
    }


def analysis1_assert_execution(log_results: list[LogResult]) -> dict:
    """
    For files with has_real_calls=True:
    Did the assertions execute, or did the test fail before reaching them?
    """
    subset = [lr for lr in log_results if lr.has_real_calls]

    total_files   = len(subset)
    total_methods = sum(len(lr.method_results) for lr in subset)
    total_asserts = sum(mr.total_asserts      for lr in subset for mr in lr.method_results)
    before_fail   = sum(mr.asserts_before_fail for lr in subset for mr in lr.method_results)
    after_fail    = sum(mr.asserts_after_fail  for lr in subset for mr in lr.method_results)

    # Split into two buckets by whether any assert was reached
    executed_subset      = [lr for lr in subset if any(mr.assert_executed for mr in lr.method_results)]
    failed_before_subset = [lr for lr in subset if not any(mr.assert_executed for mr in lr.method_results)]

    # Failure kind breakdown
    kind_counts: dict[str, int] = defaultdict(int)
    for lr in subset:
        kind_counts[lr.failure_kind] += 1

    # Per-stage execution rate (upper bound)
    # exec_rate = asserts_before_fail / total_asserts
    # "before_fail" = assert source line < failure line from stack trace.
    # Upper bound because branches may skip an assert even if it appears earlier.
    stage_stats: dict[str, dict] = defaultdict(lambda: {"total": 0, "before": 0})
    for lr in subset:
        for mr in lr.method_results:
            s = stage_stats[mr.failure_kind]
            s["total"]  += mr.total_asserts
            s["before"] += mr.asserts_before_fail
    per_stage = {
        kind: {
            "total_asserts":       s["total"],
            "before_fail":         s["before"],
            "after_fail":          s["total"] - s["before"],
        }
        for kind, s in stage_stats.items()
    }

    # Exception category breakdown
    exc_categories: dict[str, int] = defaultdict(int)
    other_files: list[str] = []
    for lr in subset:
        cat = _categorize_exception(lr.failure_exception)
        exc_categories[cat] += 1
        if cat == "other":
            other_files.append(f"{lr.instance}/{lr.test_file}")

    # Overall trycatch breakdown — file level (all has_real_calls=True files)
    file_tc_total  = sum(sum(1 for c in lr.assert_calls if     c.get("in_trycatch", False)) for lr in subset)
    file_ntc_total = sum(sum(1 for c in lr.assert_calls if not c.get("in_trycatch", False)) for lr in subset)
    file_tc_before = sum(
        sum(1 for c in lr.assert_calls if c.get("in_trycatch", False)
            and lr.failure_line and c["line"] < lr.failure_line)
        for lr in subset
    )
    file_ntc_before = sum(
        sum(1 for c in lr.assert_calls if not c.get("in_trycatch", False)
            and lr.failure_line and c["line"] < lr.failure_line)
        for lr in subset
    )
    files_with_tc  = sum(1 for lr in subset if any(    c.get("in_trycatch", False) for c in lr.assert_calls))
    files_with_ntc = sum(1 for lr in subset if any(not c.get("in_trycatch", False) for c in lr.assert_calls))

    # Overall trycatch breakdown — method level
    tc_total   = sum(mr.trycatch_total           for lr in subset for mr in lr.method_results)
    tc_before  = sum(mr.trycatch_before_fail     for lr in subset for mr in lr.method_results)
    ntc_total  = sum(mr.non_trycatch_total       for lr in subset for mr in lr.method_results)
    ntc_before = sum(mr.non_trycatch_before_fail for lr in subset for mr in lr.method_results)

    return {
        # internal — not serialised to JSON
        "_subsets": {
            "assert_executed":      executed_subset,
            "failed_before_assert": failed_before_subset,
        },

        # ── File-level counts ─────────────────────────────────────────────
        "total_testfiles":                 total_files,
        "total_test_methods":              total_methods,
        "total_assert_calls":              total_asserts,
        "testfiles_where_assert_executed": len(executed_subset),
        "testfiles_failed_before_assert":  len(failed_before_subset),

        # ── Subset trycatch breakdown (file-level, one per bucket) ────────
        # before_fail = assert line < failure line (upper bound)
        # after_fail  = assert line >= failure line (definitely not reached)
        "assert_executed_trycatch_breakdown":      _file_tc_stats(executed_subset),
        "failed_before_assert_trycatch_breakdown": _file_tc_stats(failed_before_subset),"failed_before_assert_trycatch_breakdown": _file_tc_stats(failed_before_subset),

        # ── Overall failure analysis ──────────────────────────────────────
        "failure_kind_breakdown":        dict(kind_counts),
        "exception_category_breakdown":  dict(exc_categories),
        "other_files_for_investigation": other_files,
        "per_stage_assert_counts":       per_stage,

        # ── Overall trycatch breakdown (all has_real_calls=True files) ────
        # file_level   : from flat assert_calls[] — per assert call, file-scoped
        # method_level : from test_methods[]      — scoped to each @Test method
        "assert_with_trycatch_breakdown": {
            "file_level": {
                "files_with_assert_in_trycatch":      files_with_tc,
                "files_with_assert_outside_trycatch": files_with_ntc,
                "in_trycatch": {
                    "total":       file_tc_total,
                    "before_fail": file_tc_before,
                    "after_fail":  file_tc_total - file_tc_before,
                },
                "not_in_trycatch": {
                    "total":       file_ntc_total,
                    "before_fail": file_ntc_before,
                    "after_fail":  file_ntc_total - file_ntc_before,
                },
            },
            "method_level": {
                "in_trycatch": {
                    "total":       tc_total,
                    "before_fail": tc_before,
                    "after_fail":  tc_total - tc_before,
                },
                "not_in_trycatch": {
                    "total":       ntc_total,
                    "before_fail": ntc_before,
                    "after_fail":  ntc_total - ntc_before,
                },
            },
        },

        "_note": (
            "before_fail is an UPPER BOUND — assert line < failure line in source, "
            "but branches/conditions may have skipped it at runtime"
        ),
    }


# =============================================================================
# ANALYSIS 2 — has_real_calls = False
# =============================================================================

def analysis2_no_assert_failures(log_results: list[LogResult]) -> dict:
    """
    For files with has_real_calls=False:
    What caused the failure? 'other' = unknown, needs manual investigation.
    """
    subset = [lr for lr in log_results if not lr.has_real_calls]

    stage_counts:  dict[str, int] = defaultdict(int)
    exc_categories: dict[str, int] = defaultdict(int)
    other_files: list[str] = []

    for lr in subset:
        stage_counts[lr.failure_kind] += 1
        cat = _categorize_exception(lr.failure_exception)
        exc_categories[cat] += 1
        if cat == "other":
            other_files.append(f"{lr.instance}/{lr.test_file}")

    return {
        "total_testfiles":               len(subset),
        "failure_kind_breakdown":        dict(stage_counts),
        "exception_category_breakdown":  dict(exc_categories),
        "other_files_for_investigation": other_files,
        "_note": "'other' = unrecognized exception — listed for manual review",
    }


# =============================================================================
# COMBINED CONSOLE REPORT
# =============================================================================

def print_report(a1: dict, a2: dict):
    executed_subset      = a1["_subsets"]["assert_executed"]
    failed_before_subset = a1["_subsets"]["failed_before_assert"]

    W   = 70
    sep  = "=" * W
    thin = "-" * 66

    def _print_tc_block(tc_stats: dict, lrs: list[LogResult], indent: str = "    "):
        """
        Print try/catch breakdown for a subset of files.
        exec_rate (UB) = asserts with source line < failure line / total asserts.
        Upper bound because a branch may skip an assert even if it appears earlier.
        """
        tc  = tc_stats["in_trycatch"]
        ntc = tc_stats["not_in_trycatch"]
        tc_rate  = tc["before_fail"]  / tc["total"]  * 100 if tc["total"]  > 0 else 0
        ntc_rate = ntc["before_fail"] / ntc["total"] * 100 if ntc["total"] > 0 else 0

        print(f"{indent}Assert in try/catch (file-level):")
        print(f"{indent}  Files w/ ≥1 assert in try/catch : {tc_stats['files_with_assert_in_trycatch']}")
        print(f"{indent}  Files w/ ≥1 assert outside      : {tc_stats['files_with_assert_outside_trycatch']}")
        print(f"{indent}  {'Location':<28} {'Total':>6}  {'Before fail':>11}  {'After fail':>10}  {'Exec rate (UB)':>14}")
        print(f"{indent}  {'-'*66}")
        print(f"{indent}  {'Inside try/catch':<28} {tc['total']:>6}  {tc['before_fail']:>11}  {tc['after_fail']:>10}  {tc_rate:>13.1f}%")
        print(f"{indent}  {'Outside try/catch':<28} {ntc['total']:>6}  {ntc['before_fail']:>11}  {ntc['after_fail']:>10}  {ntc_rate:>13.1f}%")
        print(f"{indent}  UB = assert line < failure line; branches may still skip it")

        kind_counts: dict[str, int] = defaultdict(int)
        exc_counts:  dict[str, int] = defaultdict(int)
        for lr in lrs:
            kind_counts[lr.failure_kind] += 1
            exc_counts[_categorize_exception(lr.failure_exception)] += 1

        print(f"{indent}Failure stage:")
        for kind in _ordered_stages(kind_counts):
            cnt = kind_counts.get(kind, 0)
            if cnt == 0: continue
            print(f"{indent}  {_STAGE_LABELS.get(kind, kind):<45} {cnt:>4}")

        print(f"{indent}Failure category:")
        for cat, cnt in sorted(exc_counts.items(), key=lambda x: -x[1]):
            print(f"{indent}  {cat:<45} {cnt:>4}")

    # ── Analysis 1 ────────────────────────────────────────────────────────────
    print(f"\n{sep}")
    print(f"  ANALYSIS 1 — Files WITH assertions (has_real_calls=True)")
    print(f"{sep}")
    print(f"  Total test files                : {a1['total_testfiles']}")
    print(f"  Total @Test methods             : {a1['total_test_methods']}")
    print(f"  Total assert calls in code      : {a1['total_assert_calls']}")
    print()
    print(f"  Testfiles where assert executed : {a1['testfiles_where_assert_executed']}")
    _print_tc_block(a1["assert_executed_trycatch_breakdown"], executed_subset)
    print()
    print(f"  Testfiles failed before assert  : {a1['testfiles_failed_before_assert']}")
    _print_tc_block(a1["failed_before_assert_trycatch_breakdown"], failed_before_subset)
    print()

    print(f"  Failure lifecycle stage (all Analysis 1 files):")
    print(f"  {thin}")
    for kind in _ordered_stages(a1["failure_kind_breakdown"]):
        cnt = a1["failure_kind_breakdown"].get(kind, 0)
        if cnt == 0: continue
        print(f"  {_STAGE_LABELS.get(kind, kind):<45} {cnt:>4}")
    print()

    print(f"  Assert counts by failure stage (UB = before_fail / total):")
    print(f"  {thin}")
    for kind, s in a1["per_stage_assert_counts"].items():
        rate = s["before_fail"] / s["total_asserts"] * 100 if s["total_asserts"] > 0 else 0
        print(f"  {_STAGE_LABELS.get(kind, kind):<45} {rate:>5.1f}%  ({s['before_fail']}/{s['total_asserts']})")
    print()

    print(f"  Exception category (all Analysis 1 files):")
    print(f"  {thin}")
    for cat, cnt in sorted(a1["exception_category_breakdown"].items(), key=lambda x: -x[1]):
        print(f"  {cat:<45} {cnt:>4}")
    if a1["exception_category_breakdown"].get("other", 0) > 0:
        print(f"\n  ⚠  'other' files for manual investigation:")
        for f in a1["other_files_for_investigation"][:10]:
            print(f"       {f}")
        remaining = len(a1["other_files_for_investigation"]) - 10
        if remaining > 0:
            print(f"       ... and {remaining} more (see JSON)")

    # ── Analysis 2 ────────────────────────────────────────────────────────────
    print(f"\n{sep}")
    print(f"  ANALYSIS 2 — Files WITHOUT assertions (has_real_calls=False)")
    print(f"{sep}")
    print(f"  Total test files                : {a2['total_testfiles']}")
    print()

    print(f"  Failure lifecycle stage:")
    print(f"  {thin}")
    for kind in _ordered_stages(a2["failure_kind_breakdown"]):
        cnt = a2["failure_kind_breakdown"].get(kind, 0)
        if cnt == 0: continue
        print(f"  {_STAGE_LABELS.get(kind, kind):<45} {cnt:>4}")
    print()

    print(f"  Exception category:")
    print(f"  {thin}")
    for cat, cnt in sorted(a2["exception_category_breakdown"].items(), key=lambda x: -x[1]):
        print(f"  {cat:<45} {cnt:>4}")
    if a2["exception_category_breakdown"].get("other", 0) > 0:
        print(f"\n  ⚠  'other' files for manual investigation:")
        for f in a2["other_files_for_investigation"][:10]:
            print(f"       {f}")
        remaining = len(a2["other_files_for_investigation"]) - 10
        if remaining > 0:
            print(f"       ... and {remaining} more (see JSON)")

    print(f"\n{sep}\n")


# =============================================================================
# EXPORT FUNCTIONS
# =============================================================================

def export_analysis1_csv(log_results: list[LogResult], out_path: str):
    """Per-@Test-method rows for files with assertions."""
    subset = [lr for lr in log_results if lr.has_real_calls]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "instance", "test_file", "test_method",
            "method_start", "method_end",
            "total_asserts", "asserts_before_fail", "asserts_after_fail",
            "assert_executed",
            "failure_kind", "failure_line", "failure_in_method",
            "exception_category", "failure_exception",
            # method-level trycatch
            "method_tc_total", "method_tc_before_fail", "method_tc_after_fail",
            "method_non_tc_total", "method_non_tc_before_fail", "method_non_tc_after_fail",
            # file-level trycatch
            "file_tc_total", "file_tc_before_fail", "file_tc_after_fail",
            "file_non_tc_total", "file_non_tc_before_fail", "file_non_tc_after_fail",
        ])
        for lr in subset:
            f_tc  = [c for c in lr.assert_calls if     c.get("in_trycatch", False)]
            f_ntc = [c for c in lr.assert_calls if not c.get("in_trycatch", False)]
            fl_tc_before  = sum(1 for c in f_tc  if lr.failure_line and c["line"] < lr.failure_line)
            fl_ntc_before = sum(1 for c in f_ntc if lr.failure_line and c["line"] < lr.failure_line)
            for mr in lr.method_results:
                w.writerow([
                    lr.instance, lr.test_file, mr.method_name,
                    mr.method_start, mr.method_end,
                    mr.total_asserts, mr.asserts_before_fail, mr.asserts_after_fail,
                    mr.assert_executed,
                    mr.failure_kind, mr.failure_line, mr.failure_in_method,
                    _categorize_exception(lr.failure_exception),
                    lr.failure_exception[:120],
                    mr.trycatch_total, mr.trycatch_before_fail, mr.trycatch_after_fail,
                    mr.non_trycatch_total, mr.non_trycatch_before_fail, mr.non_trycatch_after_fail,
                    len(f_tc),  fl_tc_before,  len(f_tc)  - fl_tc_before,
                    len(f_ntc), fl_ntc_before, len(f_ntc) - fl_ntc_before,
                ])
    print(f"  CSV (Analysis 1) → {out_path}")


def export_analysis2_csv(log_results: list[LogResult], out_path: str):
    """Per-file rows for files without assertions."""
    subset = [lr for lr in log_results if not lr.has_real_calls]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "instance", "test_file",
            "failure_kind", "failure_line",
            "exception_category", "failure_exception",
        ])
        for lr in subset:
            w.writerow([
                lr.instance, lr.test_file,
                lr.failure_kind, lr.failure_line or "",
                _categorize_exception(lr.failure_exception),
                lr.failure_exception[:120],
            ])
    print(f"  CSV (Analysis 2) → {out_path}")


def export_summary_json(a1: dict, a2: dict, out_path: str):
    # _subsets holds LogResult objects; total_test_methods / total_assert_calls
    # are report-only aggregates — exclude all three from JSON serialization
    _exclude = {"_subsets", "total_test_methods", "total_assert_calls"}
    a1_serializable = {k: v for k, v in a1.items() if k not in _exclude}
    summary = {
        "analysis_1_has_real_calls_true":  a1_serializable,
        "analysis_2_has_real_calls_false": a2,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print(f"  JSON summary      → {out_path}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  Log Parser — Assertion Execution Rate Analyzer (v3)")
    print(f"{'='*70}")
    print(f"  Logs dir   : {LOGS_DIR}")
    print(f"  AST JSON   : {JSON_PATH}")
    print(f"  Output dir : {output_dir}")

    log_files = sorted(Path(LOGS_DIR).glob("*_breaking_single.log"))
    if not log_files:
        print("  No *_breaking_single.log files found.")
        sys.exit(1)

    print(f"\n  Found {len(log_files)} log files. Parsing...")
    log_results = [parse_log_file(f) for f in log_files]

    print(f"  Loading AST JSON...")
    ast_lookup  = load_ast_json(JSON_PATH)
    log_results = merge(log_results, ast_lookup)

    a1 = analysis1_assert_execution(log_results)
    a2 = analysis2_no_assert_failures(log_results)

    print_report(a1, a2)

    export_analysis1_csv(log_results, str(output_dir / "execution_rate_details.csv"))
    export_analysis2_csv(log_results, str(output_dir / "no_assert_failure_breakdown.csv"))
    export_summary_json(a1, a2,       str(output_dir / "execution_rate_summary.json"))

    print(f"\n{'='*70}")
    print(f"  Done!")
    print(f"{'='*70}\n")


if __name__ == "__main__":
    main()


  Log Parser — Assertion Execution Rate Analyzer (v3)
  Logs dir   : /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs
  AST JSON   : /Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/detected_bre_details.json
  Output dir : /Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/Assert_log_analysis

  Found 440 log files. Parsing...
  Loading AST JSON...
  ⚠  272 logs had no AST match (excluded): ['BBC04U16Test.java', 'BBC04U17Test.java', 'BBC04U21Test.java', 'BBC04U3Test.java', 'BBC06U122Test.java']
  ✓  Matched 168/440 logs to AST entries

  ANALYSIS 1 — Files WITH assertions (has_real_calls=True)
  Total test files                : 135
  Total @Test methods             : 139
  Total assert calls in code      : 241

  Testfiles where assert executed : 3
    Assert in try/catch (file-level):
      Files w/ ≥1 assert in try/catch : 3
      Files w/ ≥1 assert outside      : 0
      Location                      Total  Before fail  After fail  Exec rate (UB)
     

Sankey data: 

:Detected              #c97070
:Has Asserts           #a8b8d8
:No Asserts            #aaaaaa
:Assert Executed       #7dd4d4
:Failed Before Assert  #e8a0a0
:In Try/Catch          #d4d888

Detected [135] Has Asserts
Detected [33]  No Asserts

Has Asserts [3]   Assert Executed
Has Asserts [132] Failed Before Assert

Assert Executed [3] In Try/Catch